# 02_process_dialogues.ipynb

Post-processes the generated dialogues: extracts the dialogue text, strips punctuation, and splits each conversation into turns (`*_single_text.json`).

Paths are relative: `data/` holds dialogues and embeddings (not included). See the README.


In [ ]:
import json
import re

DATA_DIR = "data"
DATASETS = ["bank", "ill", "new_user_profile"]  # financial, medical, user profile

# The raw LLM output contains the generated dialogue; pull the dialogue block out.
pattern = r'(User:.*?(?:Assistant:.*?)*?)(?=\n\s*\nUser:|\Z)'

def extract(path_in, path_out):
    with open(path_in, "r", encoding="utf-8") as f:
        data = json.load(f)
    output, skipped = [], []
    for index, item in enumerate(data):
        match = re.findall(pattern, item["dialogue"], re.DOTALL)
        if len(match) == 1:          # no usable dialogue in this sample
            skipped.append(index)
            continue
        output.append({"privacy": item["privacy"], "dialogue": match[1].strip()})
    print(path_in, "-> skipped samples:", skipped)
    with open(path_out, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)


In [ ]:
KEEP_NUMBERS = [5.2, 6.0, 4.8, 7.5, 8.0, 5.8, 6.3, 6.2, 7.1]  # investment-return values to preserve

def clean_default(text):
    """User-profile and medical sets: replace punctuation with spaces."""
    return re.sub(r"[^\w\s]", " ", text)

def clean_finance(text):
    """Financial set: drop punctuation but keep the investment-return decimals (e.g. 5.2)."""
    out = []
    for part in re.split(r"(\d+\.\d+)", text):
        if re.fullmatch(r"\d+\.\d+", part) and float(part) in KEEP_NUMBERS:
            out.append(part)
        else:
            out.append(re.sub(r"[^\w\s]", "", part))
    return "".join(out)

def split_turns(dialogue, clean):
    history = []
    for line in dialogue.strip().split("\n"):
        for role in ("User", "Assistant"):
            if line.startswith(role + ":"):
                text = clean(line.replace(role + ":", "").strip())
                if text:
                    history.append({"role": role, "text": text})
    return history


In [ ]:
for name in DATASETS:
    extract(f"{DATA_DIR}/{name}_raw.json", f"{DATA_DIR}/{name}_text.json")
    clean = clean_finance if name == "bank" else clean_default
    with open(f"{DATA_DIR}/{name}_text.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    out = [{"privacy": it["privacy"], "dialogue": split_turns(it["dialogue"], clean)} for it in data]
    with open(f"{DATA_DIR}/{name}_single_text.json", "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print(name, len(out), "conversations")
